# Seq2Seq 모델

In [1]:
# 필요한 패키지가 없는 환경에서만 한 번 실행하세요.
# 노트북에서는 아래 명령의 주석을 해제해 실행할 수 있습니다.
# %pip install datasets sentencepiece torch tqdm

In [2]:
from datasets import load_dataset

# WMT14 독일어-영어 데이터셋을 로드합니다.
# 전체 train split을 모두 학습에 사용하지 않고, 이 중 앞쪽 12,000개만 train_subset으로 사용합니다.
# validation/test split은 평가용으로 그대로 사용합니다.
dataset = load_dataset("wmt/wmt14", "de-en")

TRAIN_SAMPLE_SIZE = 12000
train_subset = dataset["train"].select(range(min(TRAIN_SAMPLE_SIZE, len(dataset["train"]))))

print(dataset)
print("train subset size:", len(train_subset))
print("validation size:", len(dataset["validation"]))
print("test size:", len(dataset["test"]))
print(train_subset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 4508785
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3003
    })
})
train subset size: 12000
validation size: 3000
test size: 3003
{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


In [3]:
temp = dataset["train"]["translation"]
for i in temp:
    print(i)
    break

{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}


In [4]:
from pathlib import Path

# 전체 train split이 아니라 train_subset 12,000개만 사용해 SentencePiece 학습용 말뭉치를 생성합니다.
# 이렇게 하면 전체 데이터셋을 모두 쓰는 것보다 토크나이저 학습 비용과 파일 생성 시간이 줄어듭니다.
ARTIFACT_DIR = Path("sentencepiece_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

de_corpus_path = ARTIFACT_DIR / "de_corpus_train12000.txt"
en_corpus_path = ARTIFACT_DIR / "en_corpus_train12000.txt"

with open(de_corpus_path, "w", encoding="utf-8") as de_corpus, open(en_corpus_path, "w", encoding="utf-8") as en_corpus:
    for pair in train_subset["translation"]:
        de_corpus.write(pair["de"] + "\n")
        en_corpus.write(pair["en"] + "\n")

print("train subset 12,000개로 말뭉치 파일 생성 완료")
print("독일어 말뭉치:", de_corpus_path)
print("영어 말뭉치:", en_corpus_path)

train subset 12,000개로 말뭉치 파일 생성 완료
독일어 말뭉치: sentencepiece_artifacts/de_corpus_train12000.txt
영어 말뭉치: sentencepiece_artifacts/en_corpus_train12000.txt


In [5]:
import sentencepiece as spm

vocab_size = 8000
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

spm.SentencePieceTrainer.train(
    input=str(de_corpus_path),
    model_prefix=str(ARTIFACT_DIR / "encoder_spm"),
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id,
    hard_vocab_limit=False,
)

spm.SentencePieceTrainer.train(
    input=str(en_corpus_path),
    model_prefix=str(ARTIFACT_DIR / "decoder_spm"),
    vocab_size=vocab_size,
    pad_id=pad_id,
    bos_id=bos_id,
    eos_id=eos_id,
    unk_id=unk_id,
    hard_vocab_limit=False,
)

encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load(str(ARTIFACT_DIR / "encoder_spm.model"))

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load(str(ARTIFACT_DIR / "decoder_spm.model"))

sample_de_text = "Wiederaufnahme der Sitzungsperiode"
sample_en_text = "Resumption of the session"

de_ids = encoder_tokenizer.encode(sample_de_text, out_type=int)
en_ids = decoder_tokenizer.encode(sample_en_text, out_type=int)

print("독일어 토큰화:", de_ids)
print("영어 토큰화:", en_ids)
print("독일어 텍스트:", encoder_tokenizer.decode(de_ids))
print("영어 텍스트:", decoder_tokenizer.decode(en_ids))

독일어 토큰화: [2719, 702, 7, 2995]
영어 토큰화: [1049, 3693, 7, 4, 1835]
독일어 텍스트: Wiederaufnahme der Sitzungsperiode
영어 텍스트: Resumption of the session


In [6]:
de_ids = encoder_tokenizer.encode(sample_de_text)
print(type(de_ids))
print(de_ids)
[1] + de_ids + [2]

<class 'list'>
[2719, 702, 7, 2995]


[1, 2719, 702, 7, 2995, 2]

In [7]:
input_dim = len(encoder_tokenizer)
output_dim = len(decoder_tokenizer)
emb_dim = 128
hid_dim = 256
n_layers = 2
drop_ratio = 0.3
batch_size = 16
N_EPOCHS = 3
MAX_LEN = 50

In [8]:
import torch
from torch.utils.data import DataLoader, Dataset


class TranslationDataset(Dataset):
    def __init__(self, data, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN):
        self.data = data
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2

    def __len__(self):
        return len(self.data["translation"])

    def __getitem__(self, idx):
        src_text = self.data["translation"][idx]["de"]
        trg_text = self.data["translation"][idx]["en"]

        src_ids = self.encoder_tokenizer.encode(src_text)
        trg_ids = self.decoder_tokenizer.encode(trg_text)

        src_ids = src_ids[: self.max_len - 1] + [self.eos_id]
        trg_input = [self.bos_id] + trg_ids[: self.max_len - 1]
        trg_label = trg_ids[: self.max_len - 1] + [self.eos_id]

        src_ids = src_ids + [self.pad_id] * (self.max_len - len(src_ids))
        trg_input = trg_input + [self.pad_id] * (self.max_len - len(trg_input))
        trg_label = trg_label + [self.pad_id] * (self.max_len - len(trg_label))

        return torch.tensor(src_ids), torch.tensor(trg_input), torch.tensor(trg_label)


train_data = TranslationDataset(dataset["train"], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
validation_data = TranslationDataset(dataset["validation"], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
test_data = TranslationDataset(dataset["test"], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Seq2Seq without Attention

In [9]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell

In [10]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(0)
        x = self.embedding(x)
        x, (hidden, cell) = self.lstm(x, (hidden, cell))
        prediction = self.fc_out(x.squeeze(0))
        return prediction, hidden, cell

In [11]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, is_train=True, max_len=50, sos_token = 1, eos_token=2):
        # 학습 모드에서는 trg_len 사용, 추론 모드에서는 max_len까지 동적 생성
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        outputs = []

        # 인코더를 통해 context 생성
        _, hidden, cell = self.encoder(src)

        if is_train:
            for t in range(0, trg.shape[0]):
                input = trg[t]
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs.append(output.unsqueeze(0))

        else:
		    # inference에서는 target(정답)이 없기 때문에 sos_token을 생성해줍니다.
            input = torch.full((batch_size,), sos_token, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs.append(output.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1

                # 조기 종료 조건
                finished |= (top1 == eos_token)
                if finished.all():
                    break

        return outputs

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(input_dim, emb_dim, hid_dim, n_layers, drop_ratio).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, n_layers).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

In [13]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

In [14]:
from tqdm import tqdm


def train(model, data_loader, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for src, trg_input, trg_label in tqdm(data_loader):
        src = src.t().to(device)
        trg_input = trg_input.t().to(device)
        trg_label = trg_label.t().to(device)

        optimizer.zero_grad()
        outputs = model(src, trg_input, is_train=True)
        outputs = torch.cat(outputs, dim=0).reshape(-1, output_dim)
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

In [ ]:
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {train_loss:.4f}")

  6%|▋         | 18215/281800 [2:10:28<30:45:16,  2.38it/s]

# Seq2Seq with Attention

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim)

    def forward(self, src):
        # src : (src_len, batch_size)
        embedded = self.embedding(src)  # embedded : (src_len, batch_size, emb_dim)
        outputs, (hidden, cell) = self.rnn(embedded)  # outputs : (src_len, batch_size, hidden_dim)

        return outputs, hidden, cell

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, hidden_dim)
        # encoder_outputs: (src_len, batch_size, hidden_dim)

        src_len = encoder_outputs.shape[0]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim)

        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch_size, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)

        return nn.functional.softmax(attention, dim=1)  # (batch_size, src_len)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super(Decoder, self).__init__()

        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # Decoder RNN에는 embedding만 입력
        self.rnn = nn.LSTM(emb_dim, hidden_dim)
        # 출력층에는 hidden state와 attention value가 결합되어 입력
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, output_dim)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input : (batch_size,)
        # hidden : (batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim)

        input = input.unsqueeze(0)  # input : (1, batch_size)
        embedded = self.embedding(input)  # embedded : (1, batch_size, emb_dim)

        # attention distribution을 계산합니다. decoder의 이전 hidden state, s_{t-1}와 encoder의 H가 입력됩니다.
        a = self.attention(hidden[-1], encoder_outputs)  # a : (batch_size, src_len)

        # H에 가중치를 부여해 attention value(Context vector) 계산
        a = a.unsqueeze(1)  # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # encoder_outputs : (batch_size, src_len, hidden_dim)
        context = torch.bmm(a, encoder_outputs)  # context : (batch_size, 1, hidden_dim)
        context = context.permute(1, 0, 2)  # context : (1, batch_size, hidden_dim)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        # 출력층에서는 현재 hidden state와 context vector를 결합하여 예측값 생성
        output = output.squeeze(0)  # output : (batch_size, hidden_dim)
        context = context.squeeze(0)  # context : (batch_size, hidden_dim)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, cell

In [ ]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # src : (src_len, batch_size)
        # trg : (trg_len, batch_size)

        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        # 이전에는 동적으로 생성하기 위해 outputs = []를 사용했습니다.
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)

        # Encoder는 동일합니다.
        encoder_outputs, hidden, cell = self.encoder(src)

        for t in range(trg_len):
            input = trg[t]
            # 기존에는 이전 cell의 추론 값(input), hidden state(hidden, cell)만 입력했다면 encoder_outputs(H)가 추가됩니다.
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[t] = output

        # 인퍼런스 부분은 생략합니다.
        return outputs

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
attention = BahdanauAttention(hid_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, attention).to(device)
model = Seq2SeqAttention(encoder, decoder).to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

In [ ]:
def train(model, data_loader, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for src, trg_input, trg_label in tqdm(data_loader):
        src = src.t().to(device)
        trg_input = trg_input.t().to(device)
        trg_label = trg_label.t().to(device)

        optimizer.zero_grad()
        outputs = model(src, trg_input)
        outputs = outputs.reshape(-1, outputs.shape[-1])
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

In [ ]:
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {train_loss:.4f}")

100%|██████████| 1/1 [00:01<00:00,  1.04s/it]


Epoch 1/3, Train Loss: 4.2920


100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


Epoch 2/3, Train Loss: 4.2049


100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Epoch 3/3, Train Loss: 4.1102


# 하이퍼파라미터 튜닝 실험 및 결과 비교

위쪽 셀에서는 WMT14 독일어-영어 데이터셋을 로드한 뒤, 전체 train split 중 12,000개만 `train_subset`으로 선택했습니다. 여기서는 이 12,000개 학습 샘플과 원래 validation/test split을 사용해 기본 순차-순차 모델과 어텐션 모델을 비교합니다.

이렇게 진행하는 이유는 다음과 같습니다.

- 전체 train split을 모두 사용하면 학습 시간이 매우 길어지므로, 실험 가능한 크기인 12,000개로 제한합니다.
- 미니 데이터셋보다는 훨씬 많은 문장 쌍을 사용하므로 모델 구조 차이를 더 현실적으로 비교할 수 있습니다.
- validation/test split은 그대로 유지해 학습에 사용하지 않은 데이터에서 일반화 성능을 확인합니다.
- 결과는 `results_train12000/` 폴더에 저장해 기존 미니 실험 결과와 전체 데이터셋 실험 결과와 구분합니다.

주의: 12,000개만 사용해도 CPU에서는 시간이 걸릴 수 있습니다. 가능하면 GPU 환경에서 실행하는 것이 좋습니다.

## 튜닝 후보와 선택 기준

이번 실험에서 실제로 바꾼 하이퍼파라미터는 모델 종류, 임베딩 차원, 은닉 상태 차원입니다. CPU 환경에서 빠르게 비교하기 위해 층 수, 드롭아웃, 학습률은 우선 고정했습니다.

| run | 모델 | 임베딩 차원 | 은닉 상태 차원 | 층 수 | 드롭아웃 | 학습률 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| 1 | seq2seq | 32 | 64 | 1 | 0.0 | 0.003 |
| 2 | seq2seq | 64 | 96 | 1 | 0.0 | 0.003 |
| 3 | 어텐션 | 32 | 64 | 1 | 0.0 | 0.003 |
| 4 | 어텐션 | 64 | 96 | 1 | 0.0 | 0.003 |

선택 기준은 검증 손실입니다. 학습 손실이 아니라 검증 손실을 기준으로 삼은 이유는, 학습 데이터에만 잘 맞는 모델보다 처음 보는 데이터에서도 손실이 낮은 모델을 선택하기 위해서입니다.

### 추가로 튜닝하면 좋은 항목

이번 실험은 빠른 재현을 위해 작은 탐색만 수행했습니다. 더 정교한 비교를 하려면 아래 항목들을 추가로 튜닝하는 것이 좋습니다.

| 추가로 튜닝하면 좋은 항목 | 이유 |
| --- | --- |
| learning rate | 학습 속도와 안정성에 큰 영향을 줍니다. 너무 크면 손실이 불안정하게 튀고, 너무 작으면 학습이 느리거나 충분히 수렴하지 않을 수 있습니다. |
| batch size | gradient 안정성과 일반화에 영향을 줍니다. 작은 batch는 노이즈가 커서 일반화에 도움이 될 수 있지만 학습이 불안정할 수 있고, 큰 batch는 안정적이지만 일반화가 떨어질 수 있습니다. |
| dropout | 과적합 방지에 중요합니다. 데이터가 작을수록 모델이 학습 데이터에 과하게 맞춰질 수 있으므로 dropout 후보를 비교할 필요가 있습니다. |
| layer 수 | 모델 표현력에 영향을 줍니다. layer가 많으면 더 복잡한 패턴을 학습할 수 있지만, 데이터가 작으면 과적합이나 학습 불안정이 생길 수 있습니다. |
| epoch 수 | 충분히 학습되었는지 확인하는 데 필요합니다. epoch가 적으면 과소학습, 너무 많으면 과적합이 발생할 수 있으므로 검증 손실 추이를 함께 봐야 합니다. |
| teacher forcing ratio | 순차-순차 생성 성능에 영향을 줍니다. 학습 중 정답 토큰을 얼마나 자주 decoder 입력으로 넣을지 결정하며, 추론 시 실제 생성 방식과의 차이를 줄이는 데 중요합니다. |

따라서 확장 실험을 한다면 현재의 `model_type`, `emb_dim`, `hidden_dim` 비교에 더해 `learning rate`, `batch size`, `dropout`, `layer 수`, `epoch 수`, `teacher forcing ratio`까지 포함한 grid search 또는 random search로 확장할 수 있습니다.

In [ ]:
# train 데이터 12,000개 기반 하이퍼파라미터 튜닝 실험 코드
import csv
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3


class FullTranslationDataset(Dataset):
    def __init__(self, split_data, encoder_tokenizer, decoder_tokenizer, max_len):
        self.data = split_data
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data["translation"])

    def _pad(self, ids):
        return ids[: self.max_len] + [PAD_ID] * max(0, self.max_len - len(ids))

    def __getitem__(self, idx):
        pair = self.data["translation"][idx]
        src_ids = self.encoder_tokenizer.encode(pair["de"])[: self.max_len - 1] + [EOS_ID]
        trg_ids = self.decoder_tokenizer.encode(pair["en"])[: self.max_len - 1]
        trg_input = [BOS_ID] + trg_ids
        trg_label = trg_ids + [EOS_ID]
        return (
            torch.tensor(self._pad(src_ids)),
            torch.tensor(self._pad(trg_input)),
            torch.tensor(self._pad(trg_label)),
        )


class TuneEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0.0)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell


class TuneDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, token, hidden, cell):
        embedded = self.embedding(token.unsqueeze(0))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        return self.fc_out(output.squeeze(0)), hidden, cell


class TuneSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        trg_len, batch_size = trg.shape
        outputs = torch.zeros(trg_len, batch_size, self.decoder.output_dim, device=src.device)
        _, hidden, cell = self.encoder(src)
        for t in range(trg_len):
            outputs[t], hidden, cell = self.decoder(trg[t], hidden, cell)
        return outputs


class TuneBahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)


class TuneAttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.attention = TuneBahdanauAttention(hidden_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, token, hidden, cell, encoder_outputs):
        embedded = self.embedding(token.unsqueeze(0))
        attn = self.attention(hidden[-1], encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn, encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        pred = self.fc_out(torch.cat((output.squeeze(0), context.squeeze(0)), dim=1))
        return pred, hidden, cell


class TuneSeq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        trg_len, batch_size = trg.shape
        outputs = torch.zeros(trg_len, batch_size, self.decoder.output_dim, device=src.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        for t in range(trg_len):
            outputs[t], hidden, cell = self.decoder(trg[t], hidden, cell, encoder_outputs)
        return outputs


def make_tune_model(config, input_dim, output_dim, device):
    model_type, emb_dim, hidden_dim, n_layers, dropout, lr = config
    encoder = TuneEncoder(input_dim, emb_dim, hidden_dim, n_layers, dropout)
    if model_type == "attention":
        decoder = TuneAttentionDecoder(output_dim, emb_dim, hidden_dim, n_layers, dropout)
        return TuneSeq2SeqAttention(encoder, decoder).to(device)
    decoder = TuneDecoder(output_dim, emb_dim, hidden_dim, n_layers, dropout)
    return TuneSeq2Seq(encoder, decoder).to(device)


def run_epoch(model, loader, criterion, device, optimizer=None):
    model.train(optimizer is not None)
    total_loss, total_tokens, correct_tokens = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for src, trg_input, trg_label in loader:
            src = src.t().to(device)
            trg_input = trg_input.t().to(device)
            trg_label = trg_label.t().to(device)
            if optimizer is not None:
                optimizer.zero_grad()
            outputs = model(src, trg_input)
            loss = criterion(outputs.reshape(-1, outputs.shape[-1]), trg_label.reshape(-1))
            if optimizer is not None:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            mask = trg_label != PAD_ID
            total_tokens += mask.sum().item()
            correct_tokens += ((outputs.argmax(-1) == trg_label) & mask).sum().item()
            total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    return {
        "loss": avg_loss,
        "ppl": math.exp(min(avg_loss, 20)),
        "token_acc": correct_tokens / max(total_tokens, 1),
    }


def train_one_full(config, train_loader, valid_loader, test_loader, device, epochs):
    model = make_tune_model(config, input_dim, output_dim, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config[-1])
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    best, best_state, history = None, None, []

    for epoch in range(1, epochs + 1):
        train_m = run_epoch(model, train_loader, criterion, device, optimizer)
        valid_m = run_epoch(model, valid_loader, criterion, device)
        row = {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"valid_{k}": v for k, v in valid_m.items()},
        }
        history.append(row)
        if best is None or valid_m["loss"] < best["valid_loss"]:
            best = {
                "best_epoch": epoch,
                "valid_loss": valid_m["loss"],
                "valid_ppl": valid_m["ppl"],
                "valid_token_acc": valid_m["token_acc"],
            }
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_m = run_epoch(model, test_loader, criterion, device)
    return {**best, **{f"test_{k}": v for k, v in test_m.items()}}, history


# 전체 train split 대신 train_subset 12,000개만 학습에 사용합니다.
FULL_EPOCHS = 1
FULL_BATCH_SIZE = 64
FULL_MAX_LEN = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader_full = DataLoader(
    FullTranslationDataset(train_subset, encoder_tokenizer, decoder_tokenizer, FULL_MAX_LEN),
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
)
valid_loader_full = DataLoader(
    FullTranslationDataset(dataset["validation"], encoder_tokenizer, decoder_tokenizer, FULL_MAX_LEN),
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
)
test_loader_full = DataLoader(
    FullTranslationDataset(dataset["test"], encoder_tokenizer, decoder_tokenizer, FULL_MAX_LEN),
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
)

configs = [
    ("seq2seq", 32, 64, 1, 0.0, 0.003),
    ("seq2seq", 64, 96, 1, 0.0, 0.003),
    ("attention", 32, 64, 1, 0.0, 0.003),
    ("attention", 64, 96, 1, 0.0, 0.003),
]

result_dir = Path("0701/results_train12000") if Path("0701").exists() else Path("results_train12000")
result_dir.mkdir(exist_ok=True)

summary_rows = []
start = time.time()
for run, config in enumerate(configs, 1):
    metrics, history = train_one_full(config, train_loader_full, valid_loader_full, test_loader_full, device, FULL_EPOCHS)
    model_type, emb_dim, hidden_dim, n_layers, dropout, lr = config
    row = {
        "run": run,
        "model": model_type,
        "emb_dim": emb_dim,
        "hidden_dim": hidden_dim,
        "n_layers": n_layers,
        "dropout": dropout,
        "lr": lr,
        "epochs": FULL_EPOCHS,
        "batch_size": FULL_BATCH_SIZE,
        "max_len": FULL_MAX_LEN,
        "train_size": len(train_subset),
        **metrics,
    }
    summary_rows.append(row)
    with (result_dir / f"history_run_{run}_{model_type}_e{emb_dim}_h{hidden_dim}.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

with (result_dir / "summary.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"train subset 12,000개 튜닝 완료: {time.time() - start:.1f}s")
summary_rows

## 실험 결과 확인

train 데이터 12,000개 기반 튜닝 결과는 위 코드 셀을 실행한 뒤 `results_train12000/summary.csv`에 저장됩니다. 기존 미니 데이터셋 결과나 전체 train split 결과와 혼동하지 않도록 저장 위치를 분리했습니다.

이번 코드에서 사용하는 설정은 다음과 같습니다.

| 항목 | 값 |
| --- | --- |
| 데이터셋 | WMT14 독일어-영어 |
| 학습 데이터 | 전체 train split 중 12,000개 |
| 검증 데이터 | 원래 validation split |
| 테스트 데이터 | 원래 test split |
| 토크나이저 학습 말뭉치 | train subset 12,000개 |
| epoch 수 | 1 |
| batch size | 64 |
| max length | 50 |
| 비교 모델 | 기본 순차-순차, 어텐션 순차-순차 |
| 비교 하이퍼파라미터 | 모델 종류, 임베딩 차원, 은닉 상태 차원 |

12,000개 train subset은 미니 데이터셋보다 충분히 크면서도 전체 train split보다 실행 부담이 낮습니다. 따라서 빠른 실험과 구조 비교 사이의 절충안으로 사용합니다.

In [ ]:
import pandas as pd
from pathlib import Path

summary_12000_path = Path("0701/results_train12000/summary.csv") if Path("0701").exists() else Path("results_train12000/summary.csv")

if summary_12000_path.exists():
    summary_12000 = pd.read_csv(summary_12000_path)
    display(summary_12000)
    best = summary_12000.loc[summary_12000["valid_loss"].idxmin()]
    print("검증 손실 기준 최적 조합")
    print(best)
else:
    print("아직 train subset 12,000개 튜닝 결과가 없습니다. 위 튜닝 코드 셀을 먼저 실행하세요.")

## 결과 해석 방법

train 데이터 12,000개로 실행한 뒤에는 `results_train12000/summary.csv`의 검증 손실을 기준으로 최적 조합을 선택합니다.

해석할 때는 다음 순서로 보면 됩니다.

1. `valid_loss`가 가장 낮은 run을 최적 하이퍼파라미터 조합으로 선택합니다.
2. 같은 모델 구조 안에서 임베딩 차원과 은닉 상태 차원을 키웠을 때 검증 손실이 낮아졌는지 확인합니다.
3. 같은 크기의 기본 순차-순차 모델과 어텐션 모델을 비교해 어텐션이 실제로 검증 손실을 낮췄는지 확인합니다.
4. `test_loss`, `test_ppl`, `test_token_acc`는 최종 일반화 성능을 확인하는 보조 지표로 사용합니다.

12,000개 train subset은 기존 미니 데이터셋보다 문장 다양성이 크므로 결과 신뢰도가 더 높습니다. 다만 전체 train split을 모두 사용한 것은 아니므로 최종 성능 결론이라기보다 제한된 학습 데이터에서의 비교로 해석해야 합니다. 더 정확한 결론을 위해서는 epoch 수, learning rate, dropout, teacher forcing ratio를 추가로 튜닝하고, 가능하면 더 많은 train 샘플을 사용하는 것이 좋습니다.